# 07 · Alert explanations for analysts + dashboard data

An analyst opening an alert needs to know **why** it fired, in plain English, e.g.
*"Amount 8.2x this customer's usual | £1,950 spent in last 24h | Night-time (23:00)"*.

Each reason's strength = how rare that value is among **genuine** 2019 transactions (rarer = stronger). The top 3 reasons are shown.
Then we export tidy tables for the Power BI dashboard.

In [ ]:
# 1. Load test-period scores, the policy, and genuine 2019 behaviour as the yardstick
import json, os
import numpy as np, pandas as pd

P = json.load(open("outputs/best_policy.json"))
s = pd.read_parquet("data/clean/scores.parquet")
test = s[s["period"] == "test"].copy()
test["hour"] = test["ts"].dt.hour
test["action"] = np.select([test["score"] >= P["t_decline"], test["score"] >= P["t_step"]], ["decline", "step_up"], "approve")

f = pd.read_parquet("data/clean/features.parquet", columns=["period", "is_fraud", "category", "amt", "amt_vs_usual",
                                                             "spend_last_24h", "txn_last_1h", "secs_since_prev"])
genuine_2019 = f[(f["period"] == "train") & (f["is_fraud"] == 0)]
REF = {c: np.sort(genuine_2019[c].values) for c in ["amt", "amt_vs_usual", "spend_last_24h", "txn_last_1h", "secs_since_prev"]}
risky_cats = f[f["period"] == "train"].groupby("category")["is_fraud"].mean().nlargest(3).index.tolist()

def pct(col, v):   # share of genuine 2019 transactions at or below this value
    return np.searchsorted(REF[col], v, side="right") / len(REF[col])

print("High-risk categories (from 2019):", risky_cats)
print("Alerts (declines):", (test["action"] == "decline").sum(), "| step-ups:", (test["action"] == "step_up").sum())

In [ ]:
# 2. Build the top-3 reasons for every flagged transaction
def reasons(r):
    c = []
    if r.amt_vs_usual >= 3:
        c.append((pct("amt_vs_usual", r.amt_vs_usual), f"Amount {r.amt_vs_usual:.1f}x this customer's usual"))
    if pct("amt", r.amt) >= 0.95:
        c.append((pct("amt", r.amt), f"Large amount (£{r.amt:,.0f})"))
    if pct("spend_last_24h", r.spend_last_24h) >= 0.95:
        c.append((pct("spend_last_24h", r.spend_last_24h), f"£{r.spend_last_24h:,.0f} spent in last 24h"))
    if r.txn_last_1h >= 2:
        c.append((pct("txn_last_1h", r.txn_last_1h), f"{int(r.txn_last_1h)} other payments in the last hour"))
    if pct("secs_since_prev", r.secs_since_prev) <= 0.05:
        c.append((1 - pct("secs_since_prev", r.secs_since_prev), f"Only {r.secs_since_prev/60:.0f} min since previous payment"))
    if r.is_night == 1:
        c.append((0.90, f"Night-time ({r.hour:02d}:00)"))
    if r.category in risky_cats:
        c.append((0.85, f"High-risk category ({r.category})"))
    if r.new_merchant == 1:
        c.append((0.70, "First payment at this merchant"))
    c.sort(key=lambda x: -x[0])
    return " | ".join(t for _, t in c[:3]) if c else "Model pattern (no single rule)"

flagged = test[test["action"] != "approve"].copy()
flagged["reasons"] = flagged.apply(reasons, axis=1)
flagged["top_reason"] = flagged["reasons"].str.split(" \\| ").str[0].str.replace(r"[\d.,£]+", "#", regex=True)

cols = ["trans_num", "ts", "cc_num", "category", "amt", "age", "score", "action", "reasons", "top_reason", "is_fraud"]
flagged[cols].sort_values("score", ascending=False).to_csv("outputs/alerts_with_reasons.csv", index=False)
flagged.sort_values("score", ascending=False)[["ts", "amt", "category", "score", "action", "reasons", "is_fraud"]].head(10)

In [ ]:
# 3. Which reasons are most common, and how often is each one real fraud?
r = flagged[flagged["action"] == "decline"]
summary = (r.groupby("top_reason").agg(alerts=("is_fraud", "size"), real_fraud_pct=("is_fraud", "mean"))
             .assign(real_fraud_pct=lambda d: (d["real_fraud_pct"] * 100).round(1))
             .sort_values("alerts", ascending=False))
summary.to_csv("outputs/reason_summary.csv")
summary.head(10)

In [ ]:
# 4. Daily operations table for the dashboard (test months)
test["day"] = test["ts"].dt.normalize()
fr = test["is_fraud"] == 1
daily = test.assign(
    alert=(test["action"] == "decline").astype(int),
    stepup=(test["action"] == "step_up").astype(int),
    fraud=fr.astype(int),
    fraud_value=np.where(fr, test["amt"], 0.0),
    fraud_value_stopped=np.where(fr & (test["action"] == "decline"), test["amt"],
                         np.where(fr & (test["action"] == "step_up"), test["amt"] * P["STEPUP_STOPS_FRAUD"], 0.0)),
    false_decline=((test["action"] == "decline") & ~fr).astype(int),
).groupby("day").agg(transactions=("amt", "size"), alerts=("alert", "sum"), stepups=("stepup", "sum"),
                     frauds=("fraud", "sum"), fraud_value=("fraud_value", "sum"),
                     fraud_value_stopped=("fraud_value_stopped", "sum"), false_declines=("false_decline", "sum")).reset_index()

for cap in [13, 21]:     # backlog if analysts can review `cap` alerts per day
    b, out = 0, []
    for a in daily["alerts"]:
        b = max(0, b + a - cap); out.append(b)
    daily[f"backlog_cap{cap}"] = out
daily.to_csv("outputs/pbi_daily_ops.csv", index=False)
print(daily[["transactions", "alerts", "stepups", "frauds", "fraud_value", "fraud_value_stopped", "false_declines"]].sum().round(0))
daily.head()